# DFT spectroscopic holdout CV

Empirical check that the **DFT global XRR model** generalizes along photon energy when trained only on two anchor energies. This supports interpreting a large BIC margin over the free-tensor model as predictive parsimony, not only parameter counting.

**Protocol**

- **Always fit:** 250 eV (off-resonance) and 283.7 eV (edge anchor)
- **Holdout A:** resonant energies with `E > 283.7` eV
- **Holdout B:** pre-resonant energies with `250 < E < 283.7` eV

**Out of scope:** free-tensor CV (per-energy OOC degrees of freedom are not spectroscopically predictive without refitting each energy).

**Comparison:** holdout chi-squared from the 2-energy train fit vs per-energy chi-squared from the full joint DFT pickle at the same energies.


In [ ]:
import copy

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyref.fitting as fit
from IPython.display import display

from utils import models_root, read_fit, read_xrr
from utils.helpers.fitting_helper import ObjectiveInterface, bic, reduced_chi2
from utils.helpers.plotting_helper import set_plotting_defaults

set_plotting_defaults()

NOTEBOOK_DIR = models_root.parent / "notebooks" / "manuscript"
FIT_ENERGIES = (250.0, 283.7)
RESONANT_EDGE = 283.7
REFIT_TRAIN = True

In [ ]:
def ndata_objective(obj: fit.Objective) -> int:
    return len(obj.data.s.x) + len(obj.data.p.x)


def objective_at_energy(global_obj: fit.GlobalObjective, energy: float) -> fit.Objective:
    for o in global_obj.objectives:
        if float(o.model.energy) == float(energy):
            return o
    raise KeyError(f"Energy {energy} eV not in global objective")


def subset_global(
    global_obj: fit.GlobalObjective, energies: tuple[float, ...]
) -> fit.GlobalObjective:
    want = {float(e) for e in energies}
    objs = [o for o in global_obj.objectives if float(o.model.energy) in want]
    have = {float(o.model.energy) for o in objs}
    missing = want - have
    if missing:
        raise ValueError(f"Missing energies in subset: {sorted(missing)}")
    return fit.GlobalObjective(objs)


def parameter_map(obj: fit.Objective | fit.GlobalObjective) -> dict[str, object]:
    return {p.name: p for p in obj.varying_parameters()}


def apply_parameters_by_name(
    source: fit.Objective | fit.GlobalObjective,
    target: fit.Objective | fit.GlobalObjective,
) -> int:
    src = parameter_map(source)
    n_mapped = 0
    for p in target.varying_parameters():
        q = src.get(p.name)
        if q is None:
            continue
        p.setp(value=float(q.value), vary=False)
        n_mapped += 1
    return n_mapped


def polish_global(global_obj: fit.GlobalObjective) -> fit.GlobalObjective:
    obj = copy.deepcopy(global_obj)
    fitter = fit.CurveFitter(obj)
    fitter.fit("SLSQP")
    return obj


def holdout_energy_sets(all_energies: list[float]) -> dict[str, list[float]]:
    fit_set = set(FIT_ENERGIES)
    return {
        "resonant_holdout": sorted(e for e in all_energies if e > RESONANT_EDGE),
        "preresonant_holdout": sorted(
            e for e in all_energies if (e < RESONANT_EDGE and e not in fit_set)
        ),
    }


def region_label(en: float) -> str:
    if en in FIT_ENERGIES:
        return "fit"
    if en > RESONANT_EDGE:
        return "resonant_holdout"
    if en > FIT_ENERGIES[0]:
        return "preresonant_holdout"
    return "other"


def run_dft_cv_mode(
    full: fit.GlobalObjective,
    mode: str,
    val_energies: list[float],
) -> pd.DataFrame:
    train = subset_global(full, FIT_ENERGIES)
    if REFIT_TRAIN:
        train = polish_global(train)

    rows = []
    for en in val_energies:
        o_pred = copy.deepcopy(objective_at_energy(full, en))
        apply_parameters_by_name(train, o_pred)
        o_joint = objective_at_energy(full, en)
        nd = ndata_objective(o_pred)
        chi2_h = float(o_pred.chisqr())
        chi2_j = float(o_joint.chisqr())
        rows.append(
            {
                "cv_mode": mode,
                "energy_eV": en,
                "ndata": nd,
                "chi2_holdout_2E": chi2_h,
                "chi2_full_joint": chi2_j,
                "rms_holdout_2E": np.sqrt(chi2_h / nd),
                "rms_full_joint": np.sqrt(chi2_j / nd),
                "rms_ratio_holdout_to_joint": np.sqrt(chi2_h / nd) / np.sqrt(chi2_j / nd),
                "region": region_label(en),
            }
        )
    return pd.DataFrame(rows)


def summarize_mode(df: pd.DataFrame) -> pd.Series:
    return pd.Series(
        {
            "n_energies": len(df),
            "sum_chi2_holdout": df["chi2_holdout_2E"].sum(),
            "sum_chi2_joint": df["chi2_full_joint"].sum(),
            "mean_rms_holdout": df["rms_holdout_2E"].mean(),
            "mean_rms_joint": df["rms_full_joint"].mean(),
            "mean_rms_ratio": df["rms_ratio_holdout_to_joint"].mean(),
            "max_rms_ratio": df["rms_ratio_holdout_to_joint"].max(),
        }
    )

In [ ]:
dft_full = read_fit("dft/dft_en_offset_new2.pkl", material="znpc", source="local")
_ = read_xrr("reflectivity_data", material="znpc")

all_energies = sorted({float(o.model.energy) for o in dft_full.objectives})
holdout_sets = holdout_energy_sets(all_energies)
holdout_sets

In [ ]:
stats_path = NOTEBOOK_DIR / "model_stats.csv"
model_stats = pd.read_csv(stats_path) if stats_path.exists() else None

dft_ctx = pd.DataFrame(
    [
        {
            "model": "dft_full_joint",
            "nparams": len(dft_full.varying_parameters()),
            "ndata": ObjectiveInterface(dft_full).ndata(),
            "chi2": float(dft_full.chisqr()),
            "bic": bic(dft_full),
            "reduced_chi2": reduced_chi2(dft_full),
        }
    ]
)
display(dft_ctx)

if model_stats is not None:
    free_row = model_stats.loc[
        model_stats["model"] == "fitting_results_free_model_2"
    ].iloc[0]
    delta_bic = float(free_row["bic"]) - float(dft_ctx["bic"].iloc[0])
    bayes_factor_dft = float(np.exp(-0.5 * delta_bic))
    print(
        f"free_model_2 vs dft_full_joint: Delta_BIC = {delta_bic:.1f}, "
        f"BF_10 (DFT preferred) ~ {bayes_factor_dft:.2e}"
    )

In [ ]:
dft_train = subset_global(dft_full, FIT_ENERGIES)
if REFIT_TRAIN:
    dft_train = polish_global(dft_train)

pd.DataFrame(
    [
        {
            "chi2_train": float(dft_train.chisqr()),
            "reduced_chi2_train": reduced_chi2(dft_train),
            "nparams_train": len(dft_train.varying_parameters()),
            "ndata_train": ObjectiveInterface(dft_train).ndata(),
        }
    ]
)

In [ ]:
cv_frames = [run_dft_cv_mode(dft_full, mode, val_E) for mode, val_E in holdout_sets.items()]
cv = pd.concat(cv_frames, ignore_index=True).sort_values(["cv_mode", "energy_eV"])
summary = (
    cv.groupby("cv_mode", as_index=False)
    .apply(summarize_mode, include_groups=False)
    .reset_index(drop=True)
)
cv, summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(7.0, 3.0), sharey=True)

for ax, mode in zip(axes, holdout_sets.keys(), strict=True):
    sub = cv.query("cv_mode == @mode").sort_values("energy_eV")
    ratio_mean = float(summary.loc[summary["cv_mode"] == mode, "mean_rms_ratio"].iloc[0])
    ax.plot(
        sub["energy_eV"],
        sub["rms_holdout_2E"],
        "o-",
        label="2E train (predictive)",
        ms=4,
    )
    ax.plot(
        sub["energy_eV"],
        sub["rms_full_joint"],
        "s--",
        label="full joint (in-sample)",
        ms=4,
    )
    ax.set_xlabel("Photon energy (eV)")
    ax.set_title(f"{mode.replace('_', ' ')}\nmean RMS ratio = {ratio_mean:.2f}")
    ax.legend(frameon=False)

axes[0].set_ylabel(r"RMS ($\sqrt{\chi^2/N}$)")
fig.tight_layout()
fig.savefig(
    NOTEBOOK_DIR / "dft_spectroscopic_cv_holdout.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

In [ ]:
for _, row in summary.iterrows():
    print(
        f"{row['cv_mode']}: mean holdout/joint RMS ratio = {row['mean_rms_ratio']:.2f} "
        f"(max {row['max_rms_ratio']:.2f})"
    )
print(
    "Interpretation: ratios moderately above 1 are expected when only two anchor energies "
    "constrain the global structure; the point is that tabulated OOC still yields finite "
    "predictive error on unseen energies without per-energy tensor DOFs. That empirical "
    "generality complements the astronomical Bayes factor from the full joint BIC comparison."
)

In [ ]:
cv.to_csv(NOTEBOOK_DIR / "dft_spectroscopic_cv_by_energy.csv", index=False)
summary.to_csv(NOTEBOOK_DIR / "dft_spectroscopic_cv_summary.csv", index=False)